In [25]:
import requests
import json
import pandas as pd

The initial trial in collecting data from jolpi's F1 api, where i'm solely checking for data in the current (2026) season. Given that it's only been 5 races (not including the past race in Monaco), data from past seasons from 2022 will be used to predict the standings of this season.

In [31]:
test_url = "https://api.jolpi.ca/ergast/f1/2026/results/?limit=1000"
response = requests.get(test_url)

print("Status code:", response.status_code)
print("Content-type:", response.headers.get("Content-type"))

Status code: 200
Content-type: application/json


In [32]:
data = response.json()
races = data['MRData']['RaceTable']['Races']

In [33]:
#convert json response to df
pd.json_normalize(races)

,season,round,url,raceName,date,time,Results,Circuit.circuitId,Circuit.url,Circuit.circuitName,Circuit.Location.lat,Circuit.Location.long,Circuit.Location.locality,Circuit.Location.country
0,2026,1,https://en.wikipedia.org/wiki/2026_Australian_...,Australian Grand Prix,2026-03-08,04:00:00Z,"[{'number': '63', 'position': '1', 'positionTe...",albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia
1,2026,2,https://en.wikipedia.org/wiki/2026_Chinese_Gra...,Chinese Grand Prix,2026-03-15,07:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",shanghai,https://en.wikipedia.org/wiki/Shanghai_Interna...,Shanghai International Circuit,31.3389,121.22,Shanghai,China
2,2026,3,https://en.wikipedia.org/wiki/2026_Japanese_Gr...,Japanese Grand Prix,2026-03-29,05:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",suzuka,https://en.wikipedia.org/wiki/Suzuka_Internati...,Suzuka Circuit,34.8431,136.541,Suzuka,Japan
3,2026,4,https://en.wikipedia.org/wiki/2026_Miami_Grand...,Miami Grand Prix,2026-05-03,20:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",miami,https://en.wikipedia.org/wiki/Miami_Internatio...,Miami International Autodrome,25.9581,-80.2389,Miami,USA
4,2026,5,https://en.wikipedia.org/wiki/2026_Canadian_Gr...,Canadian Grand Prix,2026-05-24,20:00:00Z,"[{'number': '12', 'position': '1', 'positionTe...",villeneuve,https://en.wikipedia.org/wiki/Circuit_Gilles_V...,Circuit Gilles Villeneuve,45.5,-73.5228,Montreal,Canada


In [34]:
test_df = pd.json_normalize(
    races,
    record_path='Results',
    meta = ['season', 'round', 'raceName', 'date', ['Circuit', 'circuitId']]
)
test_df

,number,position,positionText,points,grid,laps,status,Driver.driverId,Driver.permanentNumber,Driver.code,...,Time.millis,Time.time,FastestLap.rank,FastestLap.lap,FastestLap.Time.time,season,round,raceName,date,Circuit.circuitId
0,63,1,1,25,1,58,Finished,russell,63,RUS,...,4986801,1:23:06.801,6,21,1:22.670,2026,1,Australian Grand Prix,2026-03-08,albert_park
1,12,2,2,18,2,58,Finished,antonelli,12,ANT,...,4989775,+2.974,3,57,1:22.417,2026,1,Australian Grand Prix,2026-03-08,albert_park
2,16,3,3,15,4,58,Finished,leclerc,16,LEC,...,5002320,+15.519,5,38,1:22.579,2026,1,Australian Grand Prix,2026-03-08,albert_park
3,44,4,4,12,7,58,Finished,hamilton,44,HAM,...,5002945,+16.144,4,55,1:22.423,2026,1,Australian Grand Prix,2026-03-08,albert_park
4,1,5,5,10,6,58,Finished,norris,1,NOR,...,5038542,+51.741,2,53,1:22.358,2026,1,Australian Grand Prix,2026-03-08,albert_park
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,10,8,8,4,14,67,Lapped,gasly,10,GAS,...,5330330,+34.572,6,67,1:15.390,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
96,55,9,9,2,15,67,Lapped,sainz,55,SAI,...,5353772,+58.014,12,65,1:15.852,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
97,87,10,10,1,16,67,Lapped,bearman,87,BEA,...,5354807,+59.049,13,64,1:16.002,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
98,81,11,11,0,4,66,Lapped,piastri,81,PIA,...,5308457,+12.699,7,61,1:15.456,2026,5,Canadian Grand Prix,2026-05-24,villeneuve


Data being collected will start from the 2022 season upto the 5 rounds that have occured in the F1 season in 2026 so far. This is to reflect the ground effect era of cars that began in 2022, while also using the limited data from the 2026 season to take into account the new regulations that began this season

In [39]:
def get_data(season):
    url = f"https://api.jolpi.ca/ergast/f1/{season}/results/?limit=1000"
    response = requests.get(url).json()
    race_results = response['MRData']['RaceTable']['Races']
    return pd.json_normalize(race_results, record_path='Results', meta = ['season', 'round', 'raceName', 'date', ['Circuit', 'circuitId']]
)

In [41]:
seasons = [2022, 2023, 2024, 2025, 2026]

#forming the dataframe
df = pd.concat([get_data(season) for season in seasons], ignore_index=True)
df

,number,position,positionText,points,grid,laps,status,Driver.driverId,Driver.permanentNumber,Driver.code,...,FastestLap.rank,FastestLap.lap,FastestLap.Time.time,FastestLap.AverageSpeed.units,FastestLap.AverageSpeed.speed,season,round,raceName,date,Circuit.circuitId
0,16,1,1,26,1,57,Finished,leclerc,16,LEC,...,1,51,1:34.570,kph,206.018,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
1,55,2,2,18,3,57,Finished,sainz,55,SAI,...,3,52,1:35.740,kph,203.501,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
2,44,3,3,15,5,57,Finished,hamilton,44,HAM,...,5,53,1:36.228,kph,202.469,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
3,63,4,4,12,9,57,Finished,russell,63,RUS,...,6,56,1:36.302,kph,202.313,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
4,20,5,5,10,7,57,Finished,kevin_magnussen,20,MAG,...,8,53,1:36.623,kph,201.641,2022,1,Bahrain Grand Prix,2022-03-20,bahrain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,10,8,8,4,14,67,Lapped,gasly,10,GAS,...,6,67,1:15.390,NaN,NaN,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
496,55,9,9,2,15,67,Lapped,sainz,55,SAI,...,12,65,1:15.852,NaN,NaN,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
497,87,10,10,1,16,67,Lapped,bearman,87,BEA,...,13,64,1:16.002,NaN,NaN,2026,5,Canadian Grand Prix,2026-05-24,villeneuve
498,81,11,11,0,4,66,Lapped,piastri,81,PIA,...,7,61,1:15.456,NaN,NaN,2026,5,Canadian Grand Prix,2026-05-24,villeneuve


In [42]:
df.shape

(500, 31)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   number                         500 non-null    object
 1   position                       500 non-null    object
 2   positionText                   500 non-null    object
 3   points                         500 non-null    object
 4   grid                           500 non-null    object
 5   laps                           500 non-null    object
 6   status                         500 non-null    object
 7   Driver.driverId                500 non-null    object
 8   Driver.permanentNumber         500 non-null    object
 9   Driver.code                    500 non-null    object
 10  Driver.url                     500 non-null    object
 11  Driver.givenName               500 non-null    object
 12  Driver.familyName              500 non-null    object
 13  Drive

In [44]:
df.nunique()

number                            31
position                          22
positionText                      23
points                            17
grid                              23
laps                              49
status                            20
Driver.driverId                   32
Driver.permanentNumber            29
Driver.code                       32
Driver.url                        32
Driver.givenName                  32
Driver.familyName                 32
Driver.dateOfBirth                32
Driver.nationality                19
Constructor.constructorId         14
Constructor.url                   14
Constructor.name                  14
Constructor.nationality            7
Time.millis                      412
Time.time                        407
FastestLap.rank                   22
FastestLap.lap                    58
FastestLap.Time.time             469
FastestLap.AverageSpeed.units      1
FastestLap.AverageSpeed.speed    279
season                             5
r

In [45]:
#save the data as a csv for cleaning and processing
df.to_csv('results.csv', index=False)